[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-11-capstone-toolkit.ipynb)

# Advanced Discussion 11 — the capstone toolkit: baselines, calibration, uncertainty, slices

Reproduces the chapter 11 advanced discussion: baseline versus MLP under shift, expected calibration error, temperature scaling, deep ensembles, MC dropout, abstention and per-class error analysis.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: scikit-learn's bundled digits. Takes a few minutes on CPU (five ensemble members).

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
np.random.seed(0); torch.manual_seed(0)

X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.5, random_state=0, stratify=ytmp)
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr_t, Xva_t, Xte_t = T(Xtr), T(Xva), T(Xte); ytr_t, yva_t, yte_t = torch.tensor(ytr), torch.tensor(yva), torch.tensor(yte)

## 1. a deliberately over-confident model, and how to measure it

In [ ]:
def make(p=0.0): return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Dropout(p), nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p), nn.Linear(256, 10))
def fit(m, epochs=200, lr=2e-3, wd=0.0, seed=0):
    torch.manual_seed(seed); o = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 64):
            b = perm[i:i + 64]; o.zero_grad(); F.cross_entropy(m(Xtr_t[b]), ytr_t[b]).backward(); o.step()
    return m.eval()
def ece(probs, labels, bins=10):
    conf, pred = probs.max(1); acc = (pred == labels).float(); e = 0.0
    edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()
    return float(e)
def nll(probs, labels): return float(-torch.log(probs[torch.arange(len(labels)), labels] + 1e-12).mean())
# a noisier evaluation set: pixel noise makes the model wrong sometimes, so calibration matters
g = torch.Generator().manual_seed(0)
Xte_noisy = (Xte_t + 0.35 * torch.randn(Xte_t.shape, generator=g)).clamp(0, 1); Xva_noisy = (Xva_t + 0.35 * torch.randn(Xva_t.shape, generator=g)).clamp(0, 1)
m = fit(make())
with torch.no_grad(): lt, lv = m(Xte_noisy), m(Xva_noisy)
p = F.softmax(lt, 1)
print("single model on noisy test images: accuracy %.3f | mean confidence %.3f | ECE %.3f | NLL %.3f" % ((p.argmax(1) == yte_t).float().mean(), p.max(1).values.mean(), ece(p, yte_t), nll(p, yte_t)))

## 2. temperature scaling (fit one number on validation logits)

In [ ]:
Tp = torch.nn.Parameter(torch.ones(1)); opt = torch.optim.LBFGS([Tp], lr=0.1, max_iter=200)
def closure(): opt.zero_grad(); l = F.cross_entropy(lv / Tp, yva_t); l.backward(); return l
opt.step(closure)
p_t = F.softmax(lt / Tp.detach(), 1)
print("temperature T = %.2f -> ECE %.3f | NLL %.3f | accuracy unchanged %.3f" % (Tp.item(), ece(p_t, yte_t), nll(p_t, yte_t), (p_t.argmax(1) == yte_t).float().mean()))

## 3. deep ensemble and MC dropout

In [ ]:
members = [fit(make(), epochs=200, seed=s) for s in range(5)]
with torch.no_grad(): p_ens = torch.stack([F.softmax(mm(Xte_noisy), 1) for mm in members]).mean(0)
print("5-member ensemble: accuracy %.3f | ECE %.3f | NLL %.3f" % ((p_ens.argmax(1) == yte_t).float().mean(), ece(p_ens, yte_t), nll(p_ens, yte_t)))
md = fit(make(0.3), epochs=200); md.train()
with torch.no_grad(): p_mc = torch.stack([F.softmax(md(Xte_noisy), 1) for _ in range(30)]).mean(0)
md.eval()
with torch.no_grad(): p_det = F.softmax(md(Xte_noisy), 1)
print("dropout 0.3 model: deterministic ECE %.3f acc %.3f | MC-dropout(30) ECE %.3f acc %.3f" % (ece(p_det, yte_t), (p_det.argmax(1) == yte_t).float().mean(), ece(p_mc, yte_t), (p_mc.argmax(1) == yte_t).float().mean()))
# selective prediction: abstain on the least confident 20 %
def selective(p, labels, keep=0.8):
    conf = p.max(1).values; k = int(keep * len(conf)); idx = conf.argsort(descending=True)[:k]
    return (p[idx].argmax(1) == labels[idx]).float().mean().item()
print("accuracy on all vs on the 80%% most confident: single %.3f -> %.3f | ensemble %.3f -> %.3f" % ((p.argmax(1) == yte_t).float().mean(), selective(p, yte_t), (p_ens.argmax(1) == yte_t).float().mean(), selective(p_ens, yte_t)))

## 4. slice metrics: where does the model fail?

In [ ]:
pred = p_ens.argmax(1).numpy()
print("per-class accuracy (ensemble, noisy test):", {int(c): round(float((pred[yte == c] == c).mean()), 2) for c in range(10)})
worst = sorted(range(10), key=lambda c: (pred[yte == c] == c).mean())[:2]
print("weakest classes:", worst)

## 5. a tiny ablation table: baseline vs the model, and what each ingredient buys

In [ ]:
lr = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
rows = [("logistic regression (baseline)", lr.score(Xte, yte), lr.score(Xte_noisy.numpy(), yte))]
for name, mm in (("MLP, no dropout", fit(make())), ("MLP, dropout 0.3", fit(make(0.3))), ("MLP, weight decay 1e-3", fit(make(), wd=1e-3))):
    with torch.no_grad(): rows.append((name, (mm(Xte_t).argmax(1) == yte_t).float().mean().item(), (mm(Xte_noisy).argmax(1) == yte_t).float().mean().item()))
print("%-32s %-10s %-10s" % ("model", "clean acc", "noisy acc"))
for r in rows: print("%-32s %-10.3f %-10.3f" % r)

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.